# Session Analysis

Add more cells below as needed (one per analyze script).

**Before running:** set `DATA_FOLDER` to the folder containing your
session JSON files. This notebook expects `parse_sessions.py` (and any
`analyze_*.py` scripts you add cells for) to sit in the same folder as
this notebook, since it imports them as modules.

In [75]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

BASE_DIR = Path(r"C:\Git\CandleStateSessionAnalysis\data")
STRATEGY = "MACDTarget"  # "MACDTrail" or "MACDTarget"
DAY = "Tuesday"          # Monday / Tuesday / Wednesday / Thursday / Friday

DATA_FOLDER = BASE_DIR / STRATEGY
OUTPUT_FOLDER = DATA_FOLDER / "output" 

In [76]:
import parse_sessions

tables = parse_sessions.parse_sessions(DATA_FOLDER)
parse_sessions.save_tables(tables, OUTPUT_FOLDER)

Parsed Session_File__2026-03-02__2026-07-10_0831.json: trade_signals=768, price_levels=69, orders=0, transactions=11, positions=4, logs=1978
Parsed Session_File__2026-03-03__2026-07-10_0831.json: trade_signals=768, price_levels=99, orders=0, transactions=6, positions=3, logs=2289
Parsed Session_File__2026-03-04__2026-07-10_0831.json: trade_signals=768, price_levels=63, orders=0, transactions=11, positions=5, logs=2161
Parsed Session_File__2026-03-05__2026-07-10_0831.json: trade_signals=768, price_levels=79, orders=0, transactions=4, positions=2, logs=2636
Parsed Session_File__2026-03-06__2026-07-10_0831.json: trade_signals=768, price_levels=67, orders=0, transactions=13, positions=5, logs=2104
Parsed Session_File__2026-03-09__2026-07-10_0831.json: trade_signals=768, price_levels=90, orders=0, transactions=4, positions=2, logs=2719
Parsed Session_File__2026-03-10__2026-07-10_0831.json: trade_signals=768, price_levels=73, orders=0, transactions=4, positions=2, logs=3029
Parsed Session_Fi

In [77]:
import analyze_stats

positions, sessions = analyze_stats.load_positions_and_sessions(OUTPUT_FOLDER)
daily = analyze_stats.daily_results(positions, sessions)
trading_capital = analyze_stats.get_trading_capital(sessions)

stats = analyze_stats.compute_stats(daily, trading_capital)

print("--- Summary statistics ---")
print(analyze_stats.format_for_display(stats).to_string())

--- Summary statistics ---
StdDev              297
Average            $147
Annualized          74%
Max Loss          -$852
Total           $12,160
Target Hit %        64%


In [78]:
import analyze_DOW

positions, sessions = analyze_DOW.load_positions_and_sessions(OUTPUT_FOLDER)
daily = analyze_DOW.daily_results(positions, sessions)

print("--- Performance by day of week ---")
dow = analyze_DOW.day_of_week_summary(daily)
display(analyze_DOW.format_for_display(dow))

streaks = analyze_DOW.longest_streaks(daily)
print(f"\nLongest Win Streak:  {streaks['LongestWinStreak']}")
print(f"Longest Loss Streak: {streaks['LongestLossStreak']}")

current = analyze_DOW.current_streak(daily)
if current["Type"] is not None:
        print(f"Current Streak:      {current['Length']} {current['Type']}"
              f"{'s' if current['Length'] != 1 else ''}")


--- Performance by day of week ---


,Count,Net,Win,Loss,TargetHit,PctWin,PctTargetHit
Weekday,,,,,,,
Monday,15,"$1,361",10,5,7,67%,47%
Tuesday,18,"$4,364",16,2,13,89%,72%
Wednesday,19,"$2,464",13,6,13,68%,68%
Thursday,17,"$3,922",15,2,14,88%,82%
Friday,14,$50,7,7,6,50%,43%



Longest Win Streak:  12
Longest Loss Streak: 2
Current Streak:      1 Loss


In [79]:
import analyze_day

try:
        weekday = analyze_day.normalize_weekday(DAY)
except ValueError as e:
        print(f"Error: {e}")
else:

        positions, sessions = analyze_day.load_positions_and_sessions(OUTPUT_FOLDER)
        days = analyze_day.day_results(positions, sessions, weekday)

        if days.empty:
                print(f"No {DAY} sessions found in this data.")
        else:
                print(f"--- {len(days)} {DAY}s, sorted worst to best ---")
                print(days.to_string(index=False))

                outliers = days[days["IsOutlier"]]
                if not outliers.empty:
                        print(f"\n{len(outliers)} outlier day(s) (>1 std below the {weekday} average):")
                        print(outliers[["Date", "DailyNet"]].to_string(index=False))
                else:
                        print(f"\nNo single-day outliers -- {weekday}'s result looks spread across many days, "
                        "not driven by one or two disasters.")

                print(f"\n--- Positions from the 3 worst {weekday}s ---")
                worst = analyze_day.worst_day_positions(positions, days)
                print(worst.to_string(index=False))  

--- 18 Tuesdays, sorted worst to best ---
      Date                                     SourceFile  DailyNet  IsOutlier
2026-06-23 Session_File__2026-06-23__2026-07-10_0836.json   -342.17       True
2026-03-17 Session_File__2026-03-17__2026-07-10_0832.json    -85.44       True
2026-05-05 Session_File__2026-05-05__2026-07-10_0833.json     35.99      False
2026-05-26 Session_File__2026-05-26__2026-07-10_0834.json     86.12      False
2026-06-02 Session_File__2026-06-02__2026-07-10_0835.json    154.43      False
2026-03-10 Session_File__2026-03-10__2026-07-10_0831.json    236.34      False
2026-06-09 Session_File__2026-06-09__2026-07-10_0835.json    276.11      False
2026-04-28 Session_File__2026-04-28__2026-07-10_0833.json    288.15      False
2026-04-07 Session_File__2026-04-07__2026-07-10_0833.json    290.08      False
2026-06-16 Session_File__2026-06-16__2026-07-10_0835.json    293.59      False
2026-06-30 Session_File__2026-06-30__2026-07-10_0836.json    304.69      False
2026-03-24

In [80]:
import  analyze_symbol

positions = analyze_symbol.load_positions(OUTPUT_FOLDER)
summary = analyze_symbol.symbol_summary(positions)

print("--- Performance by symbol ---")
print(analyze_symbol.format_for_display(summary).to_string())



--- Performance by symbol ---
        Count     Net  Win  Loss PctWin
Symbol                                 
TQQQ       69  $4,609   44    25    64%
UPRO       70  $3,326   50    20    71%
SDS        68  $2,626   38    30    56%
QID        48  $1,598   22    26    46%


In [81]:
from datetime import datetime
import html as html_module
import re
import pandas as pd


def df_to_markdown_table(df, index_label=None, show_index=True):
    """
    Renders a DataFrame as a GitHub-flavored markdown table without
    needing the `tabulate` package (which may not be installed).
    show_index=False drops a plain positional index (e.g. sorted-but-not-
    meaningfully-indexed tables like `days` or `worst`).
    """
    df = df.reset_index(drop=not show_index)
    if index_label and show_index:
        df = df.rename(columns={df.columns[0]: index_label})
    header = "| " + " | ".join(str(c) for c in df.columns) + " |"
    sep = "| " + " | ".join("---" for _ in df.columns) + " |"
    rows = ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join([header, sep] + rows)


sessions_df = pd.read_csv(OUTPUT_FOLDER / "sessions.csv")

session_dates = pd.to_datetime(sessions_df["SessionStart"], utc=True).dt.tz_convert("America/New_York")
from_date = session_dates.min().strftime("%Y-%m-%d")
thru_date = session_dates.max().strftime("%Y-%m-%d")
thru_date_compact = session_dates.max().strftime("%Y%m%d")

trading_capital = analyze_stats.get_trading_capital(sessions_df)

targets = sessions_df["DayTarget"].dropna().unique()
if len(targets) > 1:
    print(f"WARNING: DayTarget varies across sessions {sorted(targets)} "
          f"-- using the first session's value ({targets[0]:,.0f}) for the report header.")
day_target = float(targets[0])

sections = [
    f"# Session Analysis Report\n\n"
    f"**Strategy:** {STRATEGY}  \n"
    f"**Period:** {from_date} to {thru_date}  \n"
    f"**Trading Capital:** ${trading_capital:,.0f}  \n"
    f"**Day Target:** ${day_target:,.0f}  \n"
    f"**Generated:** {datetime.now():%Y-%m-%d %H:%M}"
]

# --- Summary statistics ---
positions_df = pd.read_csv(OUTPUT_FOLDER / "positions.csv")
daily_net = positions_df.groupby("SourceFile", as_index=False)["RealizedGain"].sum() \
    .rename(columns={"RealizedGain": "NetValue"})
recent = sessions_df[["SourceFile", "SessionStart", "TargetHit"]].merge(daily_net, on="SourceFile", how="left")
recent["NetValue"] = recent["NetValue"].fillna(0)
recent["SessionStart_dt"] = pd.to_datetime(recent["SessionStart"], utc=True).dt.tz_convert("America/New_York")
recent = recent.sort_values("SessionStart_dt").tail(5)  # oldest-first within this 5-day window after ascending sort

recent_display = pd.DataFrame({
    "Date": recent["SessionStart_dt"].dt.strftime("%Y%m%d"),
    "NetAmount": recent["NetValue"].map(lambda v: f"-${abs(v):,.0f}" if v < 0 else f"${v:,.0f}"),
    "TargetHit": recent["TargetHit"].fillna(0).astype(int),
})

if "stats" in globals():
    stats_display = analyze_stats.format_for_display(stats).to_frame(name="Value")
    section = "## Summary Statistics\n\n" + df_to_markdown_table(stats_display, index_label="Metric")
else:
    section = "## Summary Statistics\n\n_Stats table not available -- run the analyze_stats cell first._"

if not recent_display.empty:
    section += "\n\n### Last 5 Days\n\n" + df_to_markdown_table(recent_display, show_index=False)

sections.append(section)

# --- Day of week ---
if "dow" in globals() and "streaks" in globals():
    dow_display = analyze_DOW.format_for_display(dow)
    streak_lines = (
        f"\n\n**Longest Win Streak:** {streaks['LongestWinStreak']}  \n"
        f"**Longest Loss Streak:** {streaks['LongestLossStreak']}"
    )
    if "current" in globals() and current.get("Type") is not None:
        streak_lines += (
            f"  \n**Current Streak:** {current['Length']} {current['Type']}"
            f"{'s' if current['Length'] != 1 else ''}"
        )
    sections.append(
        "## Performance by Day of Week\n\n"
        + df_to_markdown_table(dow_display, index_label="Weekday")
        + streak_lines
    )
else:
    sections.append("## Performance by Day of Week\n\n_Not available -- run the analyze_DOW cell first._")

# --- Single-day deep dive ---
""" if "weekday" in globals() and "days" in globals() and not days.empty:
    day_section = f"## {weekday} Deep Dive\n\n" + df_to_markdown_table(days, show_index=False)
    if "worst" in globals():
        day_section += f"\n\n### Positions from the 3 worst {weekday}s\n\n" + df_to_markdown_table(worst, show_index=False)
    sections.append(day_section)
else:
    sections.append("## Single-Day Deep Dive\n\n_Not available -- run the analyze_day cell first (with a valid DAY)._") """

# --- Symbol breakdown ---
if "summary" in globals():
    symbol_display = analyze_symbol.format_for_display(summary)
    sections.append("## Performance by Symbol\n\n" + df_to_markdown_table(symbol_display, index_label="Symbol"))
else:
    sections.append("## Performance by Symbol\n\n_Not available -- run the analyze_symbol cell first._")

report_md = "\n\n".join(sections)

report_md_path = OUTPUT_FOLDER / f"report_{thru_date_compact}.md"
report_md_path.write_text(report_md, encoding="utf-8")
print(f"Wrote {report_md_path}")

# --- Also render a simple standalone HTML version ---


def markdown_table_to_html(md_table: str) -> str:
    lines = [l for l in md_table.strip().splitlines() if l.strip()]
    header_cells = [c.strip() for c in lines[0].strip("|").split("|")]
    body_rows = [[c.strip() for c in l.strip("|").split("|")] for l in lines[2:]]
    thead = "<tr>" + "".join(f"<th>{html_module.escape(c)}</th>" for c in header_cells) + "</tr>"
    tbody = "".join(
        "<tr>" + "".join(f"<td>{html_module.escape(c)}</td>" for c in row) + "</tr>"
        for row in body_rows
    )
    return f"<table>\n<thead>{thead}</thead>\n<tbody>{tbody}</tbody>\n</table>"


def section_to_html(section_md: str) -> str:
    lines = section_md.split("\n\n")
    html_parts = []
    for block in lines:
        stripped = block.strip()
        if stripped.startswith("### "):
            html_parts.append(f"<h3>{html_module.escape(stripped[4:])}</h3>")
        elif stripped.startswith("## "):
            html_parts.append(f"<h2>{html_module.escape(stripped[3:])}</h2>")
        elif stripped.startswith("# "):
            html_parts.append(f"<h1>{html_module.escape(stripped[2:])}</h1>")
        elif stripped.startswith("|"):
            html_parts.append(markdown_table_to_html(stripped))
        elif stripped.startswith("_") and stripped.endswith("_"):
            html_parts.append(f"<p><em>{html_module.escape(stripped[1:-1])}</em></p>")
        else:
            # Escape first, then convert **bold** markers and markdown
            # line breaks ("  \n", two trailing spaces) to <br>.
            escaped = html_module.escape(stripped)
            escaped = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", escaped)
            escaped = escaped.replace("  \n", "<br>\n")
            html_parts.append(f"<p>{escaped}</p>")
    return "\n".join(html_parts)


body_html = "\n".join(section_to_html(s) for s in sections)
report_html = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Session Analysis Report - {html_module.escape(STRATEGY)}</title>
<style>
  body {{ font-family: -apple-system, Segoe UI, sans-serif; max-width: 900px; margin: 2rem auto; padding: 0 1rem; color: #1a1a1a; }}
  table {{ border-collapse: collapse; width: 100%; margin: 1rem 0; }}
  th, td {{ border: 1px solid #ddd; padding: 6px 10px; text-align: right; }}
  th:first-child, td:first-child {{ text-align: left; }}
  th {{ background: #f4f4f4; }}
  h1 {{ border-bottom: 2px solid #333; padding-bottom: 0.3rem; }}
  h2 {{ margin-top: 2rem; border-bottom: 1px solid #ccc; padding-bottom: 0.2rem; }}
</style>
</head>
<body>
{body_html}
</body>
</html>"""

report_html_path = OUTPUT_FOLDER / f"report_{thru_date_compact}.html"
report_html_path.write_text(report_html, encoding="utf-8")
print(f"Wrote {report_html_path}")

Wrote C:\Git\CandleStateSessionAnalysis\data\MACDTarget\output\report_20260710.md
Wrote C:\Git\CandleStateSessionAnalysis\data\MACDTarget\output\report_20260710.html
